# Library & Data import

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('Data\\311_2025_Jan_Dec.csv', low_memory=False)

In [ ]:
pd.set_option('display.max_rows', 200)  # In case rows are being cut off 

# Remodeling our data

## Removing unnecessery columns

Our research focuses on segregating different areas according to their street , the agency responsible for resolving the complaint , the resolve time itself , and the location. We chose to keep the columns below and remove the other columns since they dont provide significat information for the model in clustering the areas. We also removed Latitude and Longitude since Location column already has all the 

In [ ]:
Rem_df = df[[
    "City",
    "Street Name",
    "Incident Zip",
    "Agency",
    "Problem (formerly Complaint Type)",
    "Location Type",
    "Closed Date",
    "Created Date",
    "Status",
    "Location"
]].copy()


In [ ]:
Rem_df.head()

## Converting string type dates into datetime and calculating Resolution Time

The date in the data is of string type. Let's convert it to DateTime.

In [ ]:
# Convert to datetime
Rem_df['Closed Date'] = pd.to_datetime(Rem_df['Closed Date'], errors='coerce')
Rem_df['Created Date'] = pd.to_datetime(Rem_df['Created Date'], errors='coerce')

In [ ]:
# Creating single column representing time difference (in hours, rounded to 2 decimal places)
Rem_df["Resolution Time"] = (
    (Rem_df["Closed Date"] - Rem_df["Created Date"])
    .dt.total_seconds() / 3600
).apply(lambda x: round(x, 2) if pd.notna(x) else pd.NA).astype("Float64")

We would also like to add a column representing the day the complaint was created.

In [ ]:
Rem_df['Created Date'] = pd.to_datetime(Rem_df['Created Date'])
Rem_df['Day_Opened'] = Rem_df['Created Date'].dt.day_name()
Rem_df[['Created Date', 'Day_Opened']].head()

In [ ]:
Rem_df.head()

## Grouping same complaint types & location types & cities

In [ ]:
# We used an LLM to make a dictionary that groups the same complaint types into one.
broad_categories = {
    'Noise': [
        'Noise - Commercial', 'Noise - Residential', 'Noise', 'Noise - Street/Sidewalk', 
        'Noise - Vehicle', 'Noise - Park', 'Noise - Helicopter', 'Noise - House of Worship'
    ],
    
    'Housing & Building Maintenance': [
        'HEAT/HOT WATER', 'PLUMBING', 'FLOORING/STAIRS', 'UNSANITARY CONDITION', 'GENERAL', 
        'ELECTRIC', 'DOOR/WINDOW', 'APPLIANCE', 'Boilers', 'PAINT/PLASTER', 'ELEVATOR', 'Elevator', 
        'WATER LEAK', 'OUTSIDE BUILDING', 'Non-Residential Heat', 'Window Guard', 'Building Condition', 
        'Indoor Sewage', 'Mold'
    ],
    
    'Vehicles & Parking': [
        'Blocked Driveway', 'Derelict Vehicles', 'Illegal Parking', 'Abandoned Vehicle', 'Traffic', 
        'Abandoned Bike', 'Broken Parking Meter', 'Municipal Parking Facility', 'Bike Rack'
    ],
    
    'Street & Infrastructure': [
        'Sidewalk Condition', 'Street Light Condition', 'Street Condition', 'Root/Sewer/Sidewalk Condition', 
        'Traffic Signal Condition', 'Highway Condition', 'Bridge Condition', 'Curb Condition', 
        'Street Sign - Damaged', 'Street Sign - Missing', 'Street Sign - Dangling', 'Highway Sign - Damaged', 
        'Highway Sign - Missing', 'Highway Sign - Dangling', 'DEP Street Condition', 'DEP Sidewalk Condition', 
        'DEP Highway Condition', 'Tunnel Condition', 'Snow or Ice'
    ],
    
    'Sanitation & Trash': [
        'Dumpster Complaint', 'Dirty Condition', 'Sanitation Worker or Vehicle Complaint', 'Sewer', 
        'Illegal Dumping', 'Commercial Disposal Complaint', 'Residential Disposal Complaint', 
        'Litter Basket Complaint', 'Street Sweeping Complaint', 'Litter Basket Request', 'Graffiti',
        'Recycling Basket Complaint', 'Missed Collection', 'Institution Disposal Complaint', 
        'Transfer Station Complaint', 'DSNY Internal', 'Industrial Waste'
    ],
    
    'Animals & Pets': [
        'Dead Animal', 'Animal-Abuse', 'Unleashed Dog', 'Animal in a Park', 'Harboring Bees/Wasps', 
        'Unsanitary Animal Facility', 'Animal Facility - No Permit', 'Illegal Animal Sold', 'Pet Sale', 
        'Unsanitary Animal Pvt Property', 'Illegal Animal Kept as Pet', 'Unsanitary Pigeon Condition', 'Pet Shop'
    ],
    
    'Trees & Parks': [
        'Dead/Dying Tree', 'Damaged Tree', 'Illegal Tree Damage', 'Overgrown Tree/Branches', 
        'New Tree Request', 'Uprooted Stump', 'Violation of Park Rules', 'Plant', 'Poison Ivy', 
        'Special Natural Area District (SNAD)', 'Wood Pile Remaining'
    ],
    
    'Health & Environmental Safety': [
        'Indoor Air Quality', 'Food Establishment', 'Food Poisoning', 'Hazardous Materials', 'Lead', 
        'Asbestos', 'ASBESTOS', 'Air Quality', 'Drinking Water', 'Water Quality', 'Radioactive Material', 
        'Oil or Gas Spill', 'Cooling Tower', 'Standing Water', 'Mosquitoes', 'Building Drinking Water Tank', 
        'Water Conservation', 'Water System'
    ],
    
    'Public Order & Police': [
        'Non-Emergency Police Matter', 'Drug Activity', 'Urinating in Public', 'Smoking or Vaping', 
        'Illegal Fireworks', 'Panhandling', 'Encampment', 'Drinking', 'Disorderly Youth', 
        'Homeless Person Assistance', 'Illegal Posting', 'Posting Advertisement'
    ],
    
    'Construction & DOB': [
        'General Construction/Plumbing', 'Plumbing', 'Building/Use', 'Scaffold Safety', 'BEST/Site Safety', 
        'Construction Lead Dust', 'Cranes and Derricks', 'Construction Safety Enforcement', 'AHV Inspection Unit',
        'Special Operations', 'Borough Office'
    ],
    
    'Taxi & For-Hire Vehicles': [
        'Taxi Complaint', 'For Hire Vehicle Complaint', 'Taxi Report', 'For Hire Vehicle Report', 
        'Green Taxi Complaint', 'Taxi Compliment', 'Green Taxi Report', 'Taxi Licensee Complaint', 
        'FHV Licensee Complaint'
    ]
}

# This creates a dict that looks like {'Noise - Commercial': 'Noise', 'HEAT/HOT WATER': 'Housing & Building Maintenance', ...}
mapping_dict = {
    specific_type: broad_category 
    for broad_category, specific_types_list in broad_categories.items() 
    for specific_type in specific_types_list
}

# Applying the mapping to our dataframe
Rem_df['Complaint_Type'] = Rem_df['Problem (formerly Complaint Type)'].map(mapping_dict).fillna('Other/Misc')

# Verify the new clean categories
print(Rem_df['Complaint_Type'].value_counts())

In [ ]:
# We do the same for Location Type
# Define the categories and their specific location types
location_categories = {
    'Residential & Mixed Use': [
        'RESIDENTIAL BUILDING', 'Residential Building/House', '3+ Family Apartment Building', 
        '1-2 Family Dwelling', '3+ Family Apt. Building', 'Residence', '1-3 Family Dwelling', 
        'Apartment', 'Residential', 'Private House', 'Private Residence', 'Residential Property', 
        'Single Room Occupancy (SRO)', 'Loft Residence', '1-2 Family Mixed Use Building', 
        '3+ Family Mixed Use Building', '1-3 Family Mixed Use Building', 'Mixed Use', 
        'Mixed Use Building', 'House and Store'
    ],
    
    'Commercial & Retail': [
        'Store/Commercial', 'Club/Bar/Restaurant', 'Restaurant/Bar/Deli/Bakery', 'Business', 
        'Commercial Building', 'Retail Store', 'Commercial', 'Comercial', 'Office Building', 
        'Store', 'Restaurant', 'Catering Service', 'Tanning Salon', 'Tattoo Parlor', 'Tire Shop', 
        'Commercial Property', 'Food Establishment or Vendor', 'Mobile Food Vendor', 
        'Food Cart Vendor', 'Green Cart Vendor', 'Street Fair Vendor', 'Street Vendor', 
        'Permanent Food Stand', 'Groomer', "Veterinarian's Office", 'Sauna', 'Spa Pool', 
        'Steam Room', 'Theater', 'Sports Arena'
    ],
    
    'Street & Transportation': [
        'Street/Sidewalk', 'Street', 'Sidewalk', 'Highway', 'Bridge', 'Subway', 'Terminal', 
        'Street/Curbside', 'Ferry', 'Curb', 'Intersection', 'Alley', 'Roadway Tunnel', 'Gutter', 
        'Speed Reducer', 'Traffic Island or Median', 'Vehicle Lane', 'Subway Station', 'Crosswalk', 
        'Overpass', 'Bike Lane', 'Taxi', 'Bus Stop Shelter', 'Airport or Train/Bus Station', 
        'Street Area'
    ],
    
    'Public Spaces & Outdoors': [
        'Park', 'Park/Playground', 'Yard', 'Public/Unfenced Area', 'Lot', 'Parking Lot or Garage', 
        'Private Property', 'Vacant Lot', 'Parking Lot', 'Public Garden', 'Beach', 'Pier', 
        'Public Garden/Park', 'Public Park/Garden', 'Parking Lot/Garage', 'Vacant Lot/Property', 
        'Swamp or Pond', 'Wooded Area', 'Ground'
    ],
    
    'Institutions, Health & Services': [
        'House of Worship', 'School', 'Public School', 'Homeless Shelter', 'Soup Kitchen', 
        'School - K-12 Public', 'Government Building', 'Senior Center', 'Private School', 
        'Cafeteria - Private School', 'Cafeteria - Public School', 'Non-Profit', 'Hospital', 
        'School - College/University', 'Medical Facility', 'Cafeteria - College/University', 
        'Correctional Facility - State', 'Correctional Facility - City', 'School Safety Zone', 
        'School - K-12 Private', 'School/Pre-School', 'Government Building - Foreign', 
        'Day Care or Nursery', 'Day Care/Nursery', 'Kennel/Animal Shelter', 
        'Petting Zoo/Animal Exhibit', 'Horse Stable', 'Summer Camp'
    ],
    
    'Infrastructure & Building Components': [
        'Stairwell', 'Hallway', 'Lobby', 'Inside', 'Building Entrance', 'Roof', 'Common Area', 
        'Above Address', 'Building (Non-Residential)', 'Building', 'Abandoned Building', 
        'Vacant Building', 'Construction Site', 'Catch Basin or Sewer', 'Catch Basin/Sewer', 
        'Public Stairs', 'Loft Building - Common Areas'
    ]
}

# Flatten the dictionary to create a mapping for Pandas
location_mapping = {
    specific_loc: broad_cat 
    for broad_cat, specific_list in location_categories.items() 
    for specific_loc in specific_list
}

# Apply the mapping to the DataFrame
# We use .fillna('Other/Misc') to capture the explicit 'nan' values and any obscure locations.
Rem_df['Location Type'] = Rem_df['Location Type'].map(location_mapping).fillna('Other/Misc')

# Verify the new clean location categories
print(Rem_df['Location Type'].value_counts())

In [ ]:
# Dropping previous unorganized complaint and location type columns
Rem_df = Rem_df.drop("Problem (formerly Complaint Type)", axis = 1 , inplace = False)

In [ ]:
# This is our final table 
Rem_df.head()

Now, in our data we have a lot of cities, we would like to group them by the borough its within.

In [ ]:
# Define the categories for the 5 NYC Boroughs
# Strings are kept lowercase here to handle mixed casing in the data 
borough_categories = {
    'Queens': [
        'QUEENS','woodside', 'far rockaway', 'flushing', 'jamaica', 'woodhaven', 'south ozone park',
        'long island city', 'corona', 'astoria', 'queens village', 'springfield gardens',
        'ridgewood', 'ozone park', 'sunnyside', 'college point', 'east elmhurst', 'elmhurst',
        'howard beach', 'forest hills', 'arverne', 'jackson heights', 'oakland gardens',
        'kew gardens', 'south richmond hill', 'fresh meadows', 'hollis', 'queens', 'maspeth',
        'bayside', 'whitestone', 'little neck', 'richmond hill', 'rego park', 'bellerose',
        'middle village', 'saint albans', 'cambria heights', 'glen oaks', 'floral park',
        'rosedale', 'breezy point', 'queens(south jamaica)', 'far rockway, queens', 
        '80th street queens(ozone park)', 'laguardia airport'
    ],
    'Brooklyn': ['brooklyn','BROOKLYN'],
    'Manhattan': ['new york', 'manhattan', 'greenwich','New York', 'MANHATTAN', 'Greenwich'],
    'Bronx': ['bronx','BRONX'],
    'Staten Island': ['staten island','STATEN ISLAND']
}
# Flatten and normalize the dictionary
borough_mapping = {
    specific_city.lower().strip(): broad_borough 
    for broad_borough, specific_list in borough_categories.items() 
    for specific_city in specific_list
}
# Save original NaN mask before overwriting
original_nan_mask = Rem_df['City'].isna()
# Apply the mapping directly to the City column
Rem_df['City'] = (
    Rem_df['City']
    .astype(str)
    .str.lower()
    .str.strip()
    .map(borough_mapping)
    .fillna('Outside NYC/Misc')
)
# Restore genuine NaNs
Rem_df.loc[original_nan_mask, 'City'] = np.nan
# Verify
print(Rem_df['City'].value_counts(dropna=False))

# Downloading processed data

In [ ]:
# Let's save the table as csv
Rem_df.to_csv("Org_W_Duplicates_data_1_0.csv", index=False)